# Encoder arm v2: standardised fold scores + collapse restarts

Runtime → Change runtime type → **T4 GPU** (or L4). Then Runtime → Run all.

Changes from v1: each fold model's scores are standardised by its own out-of-fold mean and SD before pooling, and a fine-tune whose outputs collapse is restarted with a new seed (up to 2 times).

Results go to a fresh Drive folder, `MyDrive/capstone_runs_v2/`, so they never mix with the v1 run.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
REPO = 'https://github.com/ROLFFFX/ehr-inbox-triage.git'
%cd /content
!rm -rf ehr-inbox-triage && git clone -q $REPO
%cd /content/ehr-inbox-triage
!git log --oneline -1
!grep -q COLLAPSE_SD triage/methods/encoder.py && echo 'v2 code OK' || echo 'STOP: the repo does not have the v2 code yet - push it first'
!pip install -q -r requirements.txt -r requirements-gpu.txt

In [ ]:
# Keep results on Drive in a fresh v2 folder, seeded with the repo's (v2) baseline CSVs.
RUNS = '/content/drive/MyDrive/capstone_runs_v2'
!mkdir -p $RUNS/results
!rsync -a --ignore-existing --exclude archive results/ $RUNS/results/
!rm -rf results && ln -s $RUNS/results results
!ls results

In [ ]:
!python -m scripts.download_data

## 1. Quick check (~1 min): tiny model, one epoch
The tiny model usually collapses, so expect a few "restarting with a new seed" lines. That is the restart logic working. If this cell errors, stop and send me the error.

In [ ]:
!python -m scripts.run_encoders --models tiny --heads binary --epochs 1 --max-len 128 --n-boot 50 --tag smoke

## 2. Main run, seed 0 (~35 min on a T4)

In [ ]:
!python -m scripts.run_encoders --models bioclinicalbert roberta --heads binary ordinal --model-seed 0 --tag primary

## 3. Extra seeds for training variance (~70 min)

In [ ]:
for s in (1, 2):
    !python -m scripts.run_encoders --models bioclinicalbert roberta --heads binary ordinal --model-seed {s} --tag seeds/s{s}

## 4. Sensitivity definition (Levels 1–2), seed 0 (~35 min)

In [ ]:
!python -m scripts.run_encoders --models bioclinicalbert roberta --heads binary ordinal --max-level 2 --tag sens_level2

## 5. Summary and bundle
Download `encoder_results_v2.zip` from `MyDrive/capstone_runs_v2/`.

In [ ]:
import pandas as pd
for t in ['primary', 'seeds/s1', 'seeds/s2', 'sens_level2']:
    try:
        c = pd.read_csv(f'{RUNS}/results/{t}/classification.csv')
        c = c[c.method.str.startswith(('bioclinicalbert', 'roberta'))]
        print('==', t); print(c[['method', 'recall', 'fa_per_100', 'auroc', 'recall_at_ppv67']].round(3).to_string(index=False))
    except FileNotFoundError:
        print('==', t, 'missing')

In [ ]:
!cd $RUNS && zip -qr encoder_results_v2.zip results -x '*/smoke/*' && ls -lh $RUNS/encoder_results_v2.zip